In [8]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
import ssl
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
load_dotenv(override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "rag-llm-index")
PINECONE_CLOUD = os.getenv("PINECONE_CLOUD", "aws")
PINECONE_REGION = os.getenv("PINECONE_REGION", "us-east-1")
PDF_PATH = Path(os.getenv("PDF_PATH", "data.pdf"))

if not GEMINI_API_KEY:
    raise ValueError("Missing GEMINI_API_KEY in environment or .env")

if not PINECONE_API_KEY:
    raise ValueError("Missing PINECONE_API_KEY in environment or .env")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF file not found: {PDF_PATH.resolve()}")

In [11]:
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"Loaded {len(pages)} pages")

Loaded 196 pages


In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(pages)
print(f"Created {len(chunks)} chunks")

Created 686 chunks


In [14]:
pc = Pinecone(api_key=PINECONE_API_KEY)
spec = ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION)
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]
if PINECONE_INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=3072,
        metric="dotproduct",
        spec=spec,
    )
    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)
index = pc.Index(PINECONE_INDEX_NAME)

In [15]:
embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=GEMINI_API_KEY
)

text_chunks = [chunk.page_content for chunk in chunks]
metadata = [
    {"chunk_id": i, "source": PDF_PATH.name, "text": text_chunks[i]}
    for i in range(len(text_chunks))
]

embeddings = []
for i, text in enumerate(text_chunks):
    embedding = embeddings_model.embed_query(text)
    embeddings.append(embedding)
    print(f"Embedded {i+1}/{len(text_chunks)}")
    time.sleep(0.7)

Embedded 1/686
Embedded 2/686
Embedded 3/686
Embedded 4/686
Embedded 5/686
Embedded 6/686
Embedded 7/686
Embedded 8/686
Embedded 9/686
Embedded 10/686
Embedded 11/686
Embedded 12/686
Embedded 13/686
Embedded 14/686
Embedded 15/686
Embedded 16/686
Embedded 17/686
Embedded 18/686
Embedded 19/686
Embedded 20/686
Embedded 21/686
Embedded 22/686
Embedded 23/686
Embedded 24/686
Embedded 25/686
Embedded 26/686
Embedded 27/686
Embedded 28/686
Embedded 29/686
Embedded 30/686
Embedded 31/686
Embedded 32/686
Embedded 33/686
Embedded 34/686
Embedded 35/686
Embedded 36/686
Embedded 37/686
Embedded 38/686
Embedded 39/686
Embedded 40/686
Embedded 41/686
Embedded 42/686
Embedded 43/686
Embedded 44/686
Embedded 45/686
Embedded 46/686
Embedded 47/686
Embedded 48/686
Embedded 49/686
Embedded 50/686
Embedded 51/686
Embedded 52/686
Embedded 53/686
Embedded 54/686
Embedded 55/686
Embedded 56/686
Embedded 57/686
Embedded 58/686
Embedded 59/686
Embedded 60/686
Embedded 61/686
Embedded 62/686
Embedded 63/686
E

# שמירה ב-Pinecone

In [ ]:
def save_embeddings_to_pinecone(embeddings, metadata, namespace=None, batch_size=50):
    total_uploaded = 0
    try:
        for start in range(0, len(embeddings), batch_size):
            end = min(start + batch_size, len(embeddings))
            vectors = [
                {"id": f"id-{i}", "values": embeddings[i], "metadata": metadata[i]}
                for i in range(start, end)
            ]
            if namespace:
                index.upsert(vectors=vectors, namespace=namespace)
            else:
                index.upsert(vectors=vectors)
            total_uploaded += len(vectors)
            print(f"Uploaded batch {start}-{end - 1}")
        return total_uploaded
    except Exception as exc:
        raise RuntimeError("Failed to upsert vectors to Pinecone.") from exc

RuntimeError: Failed to upsert vectors to Pinecone.

In [22]:
uploaded = save_embeddings_to_pinecone(embeddings, metadata)
print(f"Uploaded {uploaded} vectors to Pinecone")

Uploaded batch 0-49
Uploaded batch 50-99
Uploaded batch 100-149
Uploaded batch 150-199
Uploaded batch 200-249
Uploaded batch 250-299
Uploaded batch 300-349
Uploaded batch 350-399
Uploaded batch 400-449
Uploaded batch 450-499
Uploaded batch 500-549
Uploaded batch 550-599
Uploaded batch 600-649
Uploaded batch 650-685
Uploaded 686 vectors to Pinecone


In [ ]:
def retrieve(query: str, top_k: int = 3):
    query_vector = embeddings_model.embed_query(query)
    
    # search Pinecone
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )
    
    print(f"\n🔍 Query: {query}\n")
    for i, match in enumerate(results["matches"]):
        print(f"--- Match {i+1} (score: {match['score']:.3f}) ---")
        print(f"Source: {match['metadata']['source']}")
        print(f"Chunk ID: {match['metadata']['chunk_id']}")
        print(f"Text: {match['metadata']['text']}")
        print()
    
    return results

In [38]:
questions = [
    "אני רוצה תרגיל בנושא של תמורות",
    "מהו עיקרון החיבור? ",
    "איך משולש פסקל קשור לנוסחאת הבינום?",
    "הסבר הפונקציה של אוילר",
    "איך משתמשים פונקציה יוצרת לחשוב מקדמים בינומיים?",
]

for q in questions:
    retrieve(q)
    time.sleep(1)


🔍 Query: אני רוצה תרגיל בנושא של תמורות

--- Match 1 (score: 0.781) ---
Source: data.pdf
Chunk ID: 87
Text: מקרה עם ביצוע הטרנספוזיציה. 
ד. נסו להוכיח ש-ג נכו...

--- Match 2 (score: 0.771) ---
Source: data.pdf
Chunk ID: 86
Text: לפני4  ;8  לפני7  ,8  לפני6  ,8  לפני4  ;7  לפני6 ...

--- Match 3 (score: 0.763) ---
Source: data.pdf
Chunk ID: 79
Text: 2.1.1  תמורות 
חליפה של  
n איברים מתוך  
n (כלומר...


🔍 Query: מהו עיקרון החיבור? 

--- Match 1 (score: 0.727) ---
Source: data.pdf
Chunk ID: 27
Text: התשובה בעמוד 141 
 
 
1.2 עקרון החיבור 
 
השאלות  ...

--- Match 2 (score: 0.723) ---
Source: data.pdf
Chunk ID: 35
Text: 2n איברים, וכך הלאה עד הקבוצה  
kA
 שבה  יש  
kn א...

--- Match 3 (score: 0.691) ---
Source: data.pdf
Chunk ID: 52
Text: ובשלב השני  4  תוצאות. אבל לא כל הצירופים של תוצאו...


🔍 Query: איך משולש פסקל קשור לנוסחאת הבינום?

--- Match 1 (score: 0.779) ---
Source: data.pdf
Chunk ID: 220
Text: פסקל בצורה אותו בו א:. הבאה   
  
0
0

     ...

--- Match 2 (score: 0.755

In [41]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

def generate_answer(query: str, top_k: int = 3):
    # שלב 1 - Retrieve
    query_vector = embeddings_model.embed_query(query)
    results = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )
    
    # שלב 2 - בניית ה-context מה-chunks
    context = "\n\n".join([
        match["metadata"]["text"] 
        for match in results["matches"]
    ])
    
    # שלב 3 - Generation
    prompt = f"""Answer the question using only the context below.
    
Context:
{context}

Question: {query}

Answer:"""

    response = client.models.generate_content(
    model="models/gemini-2.5-flash",
    contents=prompt
)
    
    print(f"\n🔍 Query: {query}\n")
    print(f"📄 Context used:\n{context[:300]}...\n")
    print(f"🤖 Answer:\n{response.text}")

In [43]:
generate_answer("איך משולש פסקל קשור לנוסחאת הבינום?")


🔍 Query: איך משולש פסקל קשור לנוסחאת הבינום?

📄 Context used:
פסקל בצורה אותו בו א:. הבאה   
  
0
0

      
    1
0

  1
1

     
   2
0

  
+ 
2
1

  
+ 
2
2

    
  3
0

  3
1

  3
2

  3
3

   
 4
0

  4
1

  4
2

  
+ 
4
3

  4
4

  
5
0

  5
1

  5
2
...

🤖 Answer:
הקשר בין משולש פסקל לנוסחת הבינום מוצג ישירות בכותרת הסעיף "3.1 נוסחת הבינום; משולש פסקל".
לאחר מכן, מוצג מבנה של משולש פסקל עם מקדמים בינומיים.
השאלות העוקבות, כמו שאלה 3.1 ו-3.2, עוסקות בפיתוח ביטויים בינומיים ובמציאת המקדמים של איברים בפיתוחים אלו, מה שמעיד על השימוש במקדמים הבינומיים של פסקל לצורך זה.
